# GEE CHIRPS Downloader - Komparasi Varian SAT vs RNL (Multi-Band Stacking)
Notebook ini didesain khusus untuk mengunduh dan menyelaraskan dua varian utama **CHIRPS v3.0** dari Google Earth Engine (GEE):
1. **CHIRPS v3 Daily SAT** (`UCSB-CHC/CHIRPS/V3/DAILY_SAT`) - Berbasis murni observasi satelit inframerah CCD.
2. **CHIRPS v3 Daily RNL** (`UCSB-CHC/CHIRPS/V3/DAILY_RNL`) - Berbasis asimilasi latar belakang model reanalisis cuaca atmosfer.

### Fitur Utama:
- **Dual Compatibility:** Dapat berjalan di environment **Lokal** maupun **Kaggle Notebook** secara otomatis.
- **Smart Cache / Input Check:** Mendeteksi dataset yang sudah diunduh sebelumnya di Kaggle Input Dataset.
- **Multi-Band Stacking (`toBands()`):** Mempercepat unduhan hingga 30x lipat tanpa *rate-limit* API.
- **Struktur Folder Terpisah:** `data/chirps_sat/` dan `data/chirps_rnl/`.

In [1]:
!pip install kagglehub geemap earthengine-api rioxarray geopandas xarray netCDF4 --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 36.0 MB/s eta 0:00:00


In [2]:
import ee
import geopandas as gpd
import geemap
from shapely.validation import make_valid
from shapely.ops import transform, unary_union
import os
import shutil
import glob
import rioxarray as rxr
import pandas as pd
import calendar
from datetime import datetime, timedelta

# ==========================================
# 0. PENGATURAN DIREKTORI DINAMIS & PARAMETER
# ==========================================
BASE_DIR = os.getcwd()

if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    file_geojson = "/kaggle/input/datasets/jerismeteo/projek-downscale/33.05_kecamatan.geojson"
    if not os.path.exists(file_geojson):
        found_geo = glob.glob("/kaggle/input/**/33.05_kecamatan.geojson", recursive=True)
        file_geojson = found_geo[0] if found_geo else os.path.join(BASE_DIR, "33.05_kecamatan.geojson")
else:
    file_geojson = os.path.join(BASE_DIR, "33.05_kecamatan.geojson")

FOLDER_OUTPUT_SAT = os.path.join(BASE_DIR, "data", "chirps_sat")
FOLDER_OUTPUT_RNL = os.path.join(BASE_DIR, "data", "chirps_rnl")

# Direktori Dataset CHIRPS yang sudah diunduh sebelumnya di Kaggle
KAGGLE_INPUT_DIR = "/kaggle/input/datasets/jerismeteo/gee-chirps-kebumen"

tahun_awal = 2000
tahun_akhir = 2026

print(f"📌 File GeoJSON        : {file_geojson}")
print(f"📌 Folder Output SAT   : {FOLDER_OUTPUT_SAT}")
print(f"📌 Folder Output RNL   : {FOLDER_OUTPUT_RNL}")
print(f"📌 Periode Pengunduhan : {tahun_awal} s.d. {tahun_akhir}")


📌 File GeoJSON        : /kaggle/input/datasets/jerismeteo/projek-downscale/33.05_kecamatan.geojson
📌 Folder Output SAT   : /kaggle/working/data/chirps_sat
📌 Folder Output RNL   : /kaggle/working/data/chirps_rnl
📌 Periode Pengunduhan : 2000 s.d. 2026


In [3]:
import json
from google.oauth2.service_account import Credentials
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    service_account_info = json.loads(user_secrets.get_secret("GEE_KEY"))
    
    SCOPES = ['https://www.googleapis.com/auth/earthengine']
    credentials = Credentials.from_service_account_info(service_account_info, scopes=SCOPES)
    
    ee.Initialize(credentials=credentials, project='staklimjerukagung')
    print("✅ Berhasil Inisialisasi GEE via Service Account (GEE_KEY)")

except Exception as e:
    print(f"⚠️ Gagal Inisialisasi via Secrets, mencoba auth manual: {e}")
    ee.Authenticate()
    ee.Initialize(project='staklimjerukagung')

✅ Berhasil Inisialisasi GEE via Service Account (GEE_KEY)


In [4]:
# ==========================================
# 2. Persiapan Batas Wilayah (GeoJSON Clean)
# ==========================================
if not os.path.exists(file_geojson):
    raise FileNotFoundError(f"File GeoJSON tidak ditemukan di: {file_geojson}")

gdf = gpd.read_file(file_geojson)
if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs("EPSG:4326")

print("Membersihkan geometri GeoJSON yang cacat...")
gdf = gdf[gdf.geometry.notna()].copy()

def _to_2d(geom):
    if geom is None or geom.is_empty: return None
    return transform(lambda x, y, z=None: (x, y), geom)

def _extract_polygonal(geom):
    if geom is None or geom.is_empty: return None
    if geom.geom_type in ("Polygon", "MultiPolygon"): return geom
    if geom.geom_type == "GeometryCollection":
        polys = [g for g in geom.geoms if g.geom_type in ("Polygon", "MultiPolygon")]
        if not polys: return None
        return unary_union(polys)
    return None

def _clean_geom(geom):
    if geom is None or geom.is_empty: return None
    geom = _to_2d(geom)
    geom = make_valid(geom)
    geom = _extract_polygonal(geom)
    if geom is None or geom.is_empty: return None
    geom = geom.buffer(0)
    if geom is None or geom.is_empty: return None
    if not geom.is_valid:
        geom = make_valid(geom)
        geom = _extract_polygonal(geom)
    if geom is None or geom.is_empty or not geom.is_valid: return None
    return geom

gdf["geometry"] = gdf["geometry"].apply(_clean_geom)
gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
gdf = gdf[gdf.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
gdf.reset_index(drop=True, inplace=True)

if gdf.empty:
    raise ValueError("Semua geometri tidak valid setelah proses cleaning.")

print("Mengonversi ke Earth Engine...")
geojson_fc = gdf.__geo_interface__
batas_kebumen = ee.FeatureCollection(geojson_fc["features"])

Membersihkan geometri GeoJSON yang cacat...
Mengonversi ke Earth Engine...


In [5]:
# ==========================================
# 3. FUNGSI UNDUH SUPER-CEPAT CHIRPS (SAT & RNL) DENGAN PRE-FLIGHT GUARDRAILS
# ==========================================
import calendar
import shutil
import xarray as xr

VARIAN_CONFIG = {
    'sat': {
        'collection': 'UCSB-CHC/CHIRPS/V3/DAILY_SAT',
        'label': 'CHIRPS v3 SAT (Satelit)',
        'folder_name': 'chirps_sat'
    },
    'rnl': {
        'collection': 'UCSB-CHC/CHIRPS/V3/DAILY_RNL',
        'label': 'CHIRPS v3 RNL (Reanalisis)',
        'folder_name': 'chirps_rnl'
    }
}

def check_file_integrity(nc_path, year, month, expected_type='chirps'):
    """
    Tier 1: Validasi Integritas File NetCDF CHIRPS.
    Memeriksa eksistensi, ukuran > 0, dimensi time, dan time steps vs jumlah hari (28-31).
    """
    if not os.path.exists(nc_path):
        return False, "File belum ada", 0
    if os.path.getsize(nc_path) == 0:
        return False, "File 0 byte (kosong/korup)", 0

    _, days_in_month = calendar.monthrange(year, month)
    expected_steps = days_in_month

    try:
        with xr.open_dataset(nc_path) as ds:
            if 'time' not in ds.dims and 'time' not in ds.coords:
                return False, "Dimensi 'time' tidak ditemukan", 0
            actual_steps = ds.sizes.get('time', len(ds['time']))
            if 'precipitation' not in ds.data_vars:
                return False, "Variabel 'precipitation' tidak ditemukan", actual_steps
            if actual_steps < expected_steps:
                return False, f"Time steps terpotong ({actual_steps}/{expected_steps} hari)", actual_steps
            return True, f"Lengkap ({actual_steps}/{expected_steps} hari)", actual_steps
    except Exception as e:
        return False, f"File rusak / korup ({e})", 0

def check_disk_space(target_dir, min_free_gb=1.5):
    """
    Tier 3: Disk Space Guardrail.
    Memeriksa sisa kapasitas penyimpanan sebelum ekspor citra.
    """
    try:
        _, _, free = shutil.disk_usage(target_dir)
        free_gb = free / (1024 ** 3)
        if free_gb < min_free_gb:
            print(f"  [DISK GUARD] Peringatan: Sisa ruang disk tinggal {free_gb:.2f} GB (< {min_free_gb} GB) di {target_dir}")
            return False, free_gb
        return True, free_gb
    except Exception:
        return True, 999.0

def cari_file_input_chirps(tipe, tahun, bulan):
    """Mengecek apakah file NetCDF CHIRPS (lengkap maupun parsial) tersedia di Input Dataset Kaggle."""
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        possible_paths = [
            f"{KAGGLE_INPUT_DIR}/data/chirps_{tipe}/{tahun}/chirps_{tipe}_{tahun}_{bulan:02d}.nc",
            f"{KAGGLE_INPUT_DIR}/data/chirps/{tahun}/chirps_{tahun}_{bulan:02d}.nc",
            f"{KAGGLE_INPUT_DIR}/{tahun}/chirps_{tipe}_{tahun}_{bulan:02d}.nc",
            f"{KAGGLE_INPUT_DIR}/chirps_{tipe}_{tahun}_{bulan:02d}.nc",
            f"/kaggle/input/datasets/jerismeteo/gee-chirps-kebumen/data/chirps_{tipe}/{tahun}/chirps_{tipe}_{tahun}_{bulan:02d}.nc"
        ]
        best_path = None
        max_steps = 0
        for p in possible_paths:
            if os.path.exists(p):
                valid, msg, steps = check_file_integrity(p, tahun, bulan, 'chirps')
                if valid:
                    return p, True, steps
                elif steps > max_steps:
                    max_steps = steps
                    best_path = p
        if best_path:
            return best_path, False, max_steps
    return None, False, 0

def unduh_chirps_varian_bulanan(tipe, tahun, bulan, batas_ee, output_base_dir, force=False):
    """
    Mengunduh presipitasi CHIRPS harian (daily) dengan 4-tier Pre-Flight Guardrails & Atomic Writing.
    """
    cfg = VARIAN_CONFIG[tipe]
    out_dir = os.path.join(output_base_dir, str(tahun))
    os.makedirs(out_dir, exist_ok=True)
    
    nc_path = os.path.join(out_dir, f"chirps_{tipe}_{tahun}_{bulan:02d}.nc")
    nc_path_tmp = f"{nc_path}.tmp"
    tif_path = os.path.join(out_dir, f"chirps_{tipe}_{tahun}_{bulan:02d}_temp.tif")
    
    # 1. Tier 1: Cek integritas file di folder kerja
    is_valid, msg, actual_steps = check_file_integrity(nc_path, tahun, bulan, 'chirps')
    if not force and is_valid:
        print(f"[{tipe.upper()} {tahun}-{bulan:02d}] [OK] File sudah lengkap ({actual_steps} hari): {nc_path} (Dilewati)")
        return nc_path
        
    # 2. Cek di Kaggle Input Datasets (Smart Partial & Full Cache)
    if not force:
        input_file, is_cache_valid, cache_steps = cari_file_input_chirps(tipe, tahun, bulan)
        if input_file and os.path.exists(input_file):
            if is_cache_valid:
                print(f"[{tipe.upper()} {tahun}-{bulan:02d}] [KAGGLE CACHE 100%] File lengkap ditemukan di {input_file}. Menyalin...")
                shutil.copy2(input_file, nc_path)
                return nc_path
            elif cache_steps > actual_steps:
                print(f"[{tipe.upper()} {tahun}-{bulan:02d}] [KAGGLE CACHE PARTIAL] Mengadopsi cache parsial ({cache_steps} hari) dari {input_file}...")
                shutil.copy2(input_file, nc_path)
                actual_steps = cache_steps

    if actual_steps > 0 and not is_valid:
        print(f"[{tipe.upper()} {tahun}-{bulan:02d}] [PERIKSA GEE] File lokal terdeteksi tidak lengkap: {msg}. Memeriksa ketersediaan citra di GEE...")
        
    tgl_mulai = f"{tahun}-{bulan:02d}-01"
    tgl_akhir = f"{tahun+1}-01-01" if bulan == 12 else f"{tahun}-{bulan+1:02d}-01"
        
    print(f"[{tipe.upper()} {tahun}-{bulan:02d}] Menarik data ImageCollection dari GEE ({cfg['label']})...")
    try:
        dataset = (ee.ImageCollection(cfg['collection'])
                   .filter(ee.Filter.date(tgl_mulai, tgl_akhir))
                   .select('precipitation'))
        
        # Tier 2: GEE Pre-Flight Metadata Check
        count = dataset.size().getInfo()
        if count == 0:
            print(f"[{tipe.upper()} {tahun}-{bulan:02d}] [INFO GEE] Tidak ada data di GEE untuk periode ini.")
            return nc_path if actual_steps > 0 else None

        if not force and actual_steps > 0 and count <= actual_steps:
            print(f"[{tipe.upper()} {tahun}-{bulan:02d}] [INFO GEE] File lokal/cache sudah up-to-date dengan data GEE saat ini ({actual_steps}/{count} citra). Dilewati.")
            return nc_path

        if actual_steps > 0:
            print(f"[{tipe.upper()} {tahun}-{bulan:02d}] [INFO GEE] Data baru ditemukan di GEE! GEE memiliki {count} citra (lokal/cache: {actual_steps}). Mengunduh pembaruan...")

        # Tier 3: Disk Space Check
        check_disk_space(out_dir)
            
        timestamps_ms = dataset.aggregate_array('system:time_start').getInfo()
        time_index = pd.to_datetime(timestamps_ms, unit='ms')
        
        stacked_image = dataset.toBands().clip(batas_ee)
        
        print(f"[{tipe.upper()} {tahun}-{bulan:02d}] Mengunduh {count} hari sekaligus ke TIF temporer...")
        geemap.ee_export_image(
            stacked_image,
            filename=tif_path,
            region=batas_ee.geometry(),
            scale=5566,
            file_per_band=False
        )
        
        # Tier 4: Atomic NetCDF Staging & Verification
        with rxr.open_rasterio(tif_path, masked=True) as da:
            da = da.rename({'band': 'time'})
            if len(da.time) == len(time_index):
                da['time'] = time_index
            else:
                da['time'] = pd.date_range(start=time_index[0], periods=len(da.time), freq='D')
                
            da.name = "precipitation"
            da.attrs["long_name"] = f"CHIRPS v3 Daily Precipitation ({tipe.upper()})"
            da.attrs["units"] = "mm/day"
            da.attrs["temporal_resolution"] = "Daily"
            da.to_netcdf(nc_path_tmp)
            
        if os.path.exists(tif_path):
            os.remove(tif_path)

        tmp_valid, tmp_msg, tmp_steps = check_file_integrity(nc_path_tmp, tahun, bulan, 'chirps')
        if tmp_valid or (tmp_steps == count):
            if os.path.exists(nc_path):
                os.remove(nc_path)
            os.replace(nc_path_tmp, nc_path)
            print(f"[{tipe.upper()} {tahun}-{bulan:02d}] [SUKSES] Tersimpan secara atomic: {nc_path} ({tmp_steps} hari)\n")
            return nc_path
        else:
            print(f"[{tipe.upper()} {tahun}-{bulan:02d}] [ERROR] Validasi atomic gagal ({tmp_msg}). File sementara dibuang.")
            if os.path.exists(nc_path_tmp):
                os.remove(nc_path_tmp)
            return None
        
    except Exception as e:
        print(f"[{tipe.upper()} {tahun}-{bulan:02d}] [ERROR]: {e}")
        if os.path.exists(tif_path): os.remove(tif_path)
        if os.path.exists(nc_path_tmp): os.remove(nc_path_tmp)
        return None

def unduh_chirps_komparasi_multi_tahun(tahun_awal, tahun_akhir, batas_ee, base_dir, force=False):
    print(f"\n{'='*70}")
    print(f"UNDUH CHIRPS v3 KOMPARASI (SAT & RNL): {tahun_awal} - {tahun_akhir} (Force: {force})")
    print(f"{'='*70}\n")
    
    folder_sat = os.path.join(base_dir, "data", "chirps_sat")
    folder_rnl = os.path.join(base_dir, "data", "chirps_rnl")
    
    for tahun in range(tahun_awal, tahun_akhir + 1):
        for bulan in range(1, 13):
            print(f"--- Periode {tahun}-{bulan:02d} ---")
            # 1. Unduh CHIRPS SAT
            unduh_chirps_varian_bulanan('sat', tahun, bulan, batas_ee, folder_sat, force=force)
            # 2. Unduh CHIRPS RNL
            unduh_chirps_varian_bulanan('rnl', tahun, bulan, batas_ee, folder_rnl, force=force)
            
    print(f"{'='*70}")
    print("SELESAI UNDUH SELURUH VARIAN CHIRPS (SAT & RNL)")
    print(f"{'='*70}")


In [6]:
# ==========================================
# 4. EKSEKUSI PENGUNDUHAN DUA VARIAN CHIRPS
# ==========================================
unduh_chirps_komparasi_multi_tahun(
    tahun_awal=tahun_awal,
    tahun_akhir=tahun_akhir,
    batas_ee=batas_kebumen,
    base_dir=BASE_DIR
)



UNDUH CHIRPS v3 KOMPARASI (SAT & RNL): 2000 - 2026 (Force: False)

--- Periode 2000-01 ---
[SAT 2000-01] [KAGGLE CACHE 100%] File lengkap ditemukan di /kaggle/input/datasets/jerismeteo/gee-chirps-kebumen/data/chirps_sat/2000/chirps_sat_2000_01.nc. Menyalin...
[RNL 2000-01] [KAGGLE CACHE 100%] File lengkap ditemukan di /kaggle/input/datasets/jerismeteo/gee-chirps-kebumen/data/chirps_rnl/2000/chirps_rnl_2000_01.nc. Menyalin...
--- Periode 2000-02 ---
[SAT 2000-02] [KAGGLE CACHE 100%] File lengkap ditemukan di /kaggle/input/datasets/jerismeteo/gee-chirps-kebumen/data/chirps_sat/2000/chirps_sat_2000_02.nc. Menyalin...
[RNL 2000-02] [KAGGLE CACHE 100%] File lengkap ditemukan di /kaggle/input/datasets/jerismeteo/gee-chirps-kebumen/data/chirps_rnl/2000/chirps_rnl_2000_02.nc. Menyalin...
--- Periode 2000-03 ---
[SAT 2000-03] [KAGGLE CACHE 100%] File lengkap ditemukan di /kaggle/input/datasets/jerismeteo/gee-chirps-kebumen/data/chirps_sat/2000/chirps_sat_2000_03.nc. Menyalin...
[RNL 2000-03] [

Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


[SAT 2026-09] [SUKSES] Tersimpan secara atomic: /kaggle/working/data/chirps_sat/2026/chirps_sat_2026_09.nc (30 hari)

[RNL 2026-09] Menarik data ImageCollection dari GEE (CHIRPS v3 RNL (Reanalisis))...
[RNL 2026-09] Mengunduh 30 hari sekaligus ke TIF temporer...
Generating URL ...
Please wait ...
Data downloaded to /kaggle/working/data/chirps_rnl/2026/chirps_rnl_2026_09_temp.tif
[RNL 2026-09] [SUKSES] Tersimpan secara atomic: /kaggle/working/data/chirps_rnl/2026/chirps_rnl_2026_09.nc (30 hari)

--- Periode 2026-10 ---
[SAT 2026-10] Menarik data ImageCollection dari GEE (CHIRPS v3 SAT (Satelit))...


Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


[SAT 2026-10] [INFO GEE] Tidak ada data di GEE untuk periode ini.
[RNL 2026-10] Menarik data ImageCollection dari GEE (CHIRPS v3 RNL (Reanalisis))...
[RNL 2026-10] [INFO GEE] Tidak ada data di GEE untuk periode ini.
--- Periode 2026-11 ---
[SAT 2026-11] Menarik data ImageCollection dari GEE (CHIRPS v3 SAT (Satelit))...
[SAT 2026-11] [INFO GEE] Tidak ada data di GEE untuk periode ini.
[RNL 2026-11] Menarik data ImageCollection dari GEE (CHIRPS v3 RNL (Reanalisis))...
[RNL 2026-11] [INFO GEE] Tidak ada data di GEE untuk periode ini.
--- Periode 2026-12 ---
[SAT 2026-12] Menarik data ImageCollection dari GEE (CHIRPS v3 SAT (Satelit))...
[SAT 2026-12] [INFO GEE] Tidak ada data di GEE untuk periode ini.
[RNL 2026-12] Menarik data ImageCollection dari GEE (CHIRPS v3 RNL (Reanalisis))...
[RNL 2026-12] [INFO GEE] Tidak ada data di GEE untuk periode ini.
SELESAI UNDUH SELURUH VARIAN CHIRPS (SAT & RNL)


In [7]:
# =========================================================================
# UPLOAD OTOMATIS KE KAGGLE DATASET (KAGGLEHUB)
# =========================================================================
import os
import glob
import json
from datetime import datetime
import kagglehub
from kagglehub import auth

def init_kaggle_auth():
    """Menginisialisasi kredensial Kaggle untuk upload versi baru dataset."""
    # 1. Cek dari Kaggle Secrets (jika running di Kaggle Kernel)
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        try:
            from kaggle_secrets import UserSecretsClient
            secrets = UserSecretsClient()
            k_user = None
            k_key = None
            for u_label in ["KAGGLE_USERNAME", "kaggle_username", "USERNAME"]:
                try:
                    k_user = secrets.get_secret(u_label)
                    if k_user: break
                except Exception:
                    pass
            for k_label in ["KAGGLE_KEY", "kaggle_key", "KAGGLE_API_KEY", "KAGGLE_TOKEN"]:
                try:
                    k_key = secrets.get_secret(k_label)
                    if k_key: break
                except Exception:
                    pass
            
            if not k_user:
                k_user = "jerismeteo"
            
            if k_key:
                os.environ["KAGGLE_USERNAME"] = k_user
                os.environ["KAGGLE_KEY"] = k_key
                auth.set_kaggle_credentials(k_user, k_key)
                print(f"[AUTH] Kredensial Kaggle aktif via Secrets untuk user: {k_user}")
                return True
        except Exception as e:
            print(f"[AUTH] Kaggle Secrets tidak tersedia: {e}")

    # 2. Cek file kaggle.json lokal
    candidates = [
        os.path.join(os.getcwd(), "kaggle.json"),
        os.path.join(os.path.dirname(os.getcwd()), "kaggle.json"),
        os.path.expanduser("~/.kaggle/kaggle.json")
    ]
    for p in candidates:
        if os.path.exists(p):
            try:
                with open(p, "r", encoding="utf-8") as f:
                    c = json.load(f)
                os.environ["KAGGLE_USERNAME"] = c["username"]
                os.environ["KAGGLE_KEY"] = c["key"]
                auth.set_kaggle_credentials(c["username"], c["key"])
                print(f"[AUTH] Kredensial Kaggle aktif via file: {p}")
                return True
            except Exception:
                pass

    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return True

    print("[AUTH] Peringatan: Kredensial Kaggle (KAGGLE_KEY) belum terkonfigurasi.")
    return False

def upload_ke_kaggle(dataset_handle, upload_root, subfolder_data, nama_dataset):
    """
    Mengunggah dataset NetCDF terbaru ke Kaggle Dataset menggunakan kagglehub.
    Menghitung jumlah file .nc valid dan mengabaikan file temporer/cache.
    """
    if not init_kaggle_auth():
        print(f"⚠️ [UPLOAD] Dilewati: Kredensial Kaggle tidak ditemukan. Tambahkan Secret 'KAGGLE_KEY' di Kaggle Secrets.")
        return False
        
    target_nc_dir = os.path.join(upload_root, subfolder_data)
    list_files = glob.glob(os.path.join(target_nc_dir, "**", "*.nc"), recursive=True)
    if not list_files:
        print(f"⚠️ [UPLOAD] Tidak ditemukan file NetCDF di: {target_nc_dir}. Upload dibatalkan.")
        return False

    now_str = datetime.now().strftime("%Y-%m-%d %H:%M WIB")
    version_notes = f"Pembaruan Otomatis Data GEE ({nama_dataset}): {len(list_files)} file NetCDF per {now_str}"

    print(f"\n{'='*75}")
    print(f"🚀 MENGUNGGAH VERSI BARU KE KAGGLE DATASET")
    print(f"🏷️  Handle Dataset : {dataset_handle}")
    print(f"📁 Folder Basis   : {upload_root}")
    print(f"📊 Total File     : {len(list_files)} file NetCDF ({subfolder_data})")
    print(f"📝 Catatan Versi  : {version_notes}")
    print(f"{'='*75}")

    try:
        kagglehub.dataset_upload(
            handle=dataset_handle,
            local_dataset_dir=upload_root,
            version_notes=version_notes,
            ignore_patterns=[
                "*.tmp", "*.tif", "*.temp", "*.geojson", "*.json",
                "*.ipynb", "__pycache__/", ".ipynb_checkpoints/"
            ]
        )
        print(f"✅ [SUKSES] Dataset {dataset_handle} berhasil diperbarui di Kaggle!")
        return True
    except Exception as e:
        print(f"❌ [GAGAL] Terjadi kesalahan saat mengunggah dataset: {e}")
        return False

# Eksekusi upload otomatis (Hanya aktif jika berjalan di Kaggle Kernel atau FORCE_UPLOAD=True)
FORCE_UPLOAD = False

if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or FORCE_UPLOAD:
    upload_ke_kaggle(
        dataset_handle="jerismeteo/gee-chirps-kebumen",
        upload_root=BASE_DIR,
        subfolder_data='data',
        nama_dataset="CHIRPS v3 Kebumen (SAT & RNL)"
    )
else:
    print("ℹ️ Eksekusi di environment lokal: Upload otomatis ke Kaggle Dataset dilewati.")
    print("💡 Untuk menguji upload lokal, ubah FORCE_UPLOAD = True.")

[AUTH] Peringatan: Kredensial Kaggle (KAGGLE_KEY) belum terkonfigurasi.
⚠️ [UPLOAD] Dilewati: Kredensial Kaggle tidak ditemukan. Tambahkan Secret 'KAGGLE_KEY' di Kaggle Secrets.
